# Preparação do dataset de segmentação

Este notebook reconstrói o dataset após a revisão manual das classes.

Estratégia adotada:

- usar **todas as imagens positivas válidas** da triagem;
- selecionar **3 imagens negativas para cada positiva**;
- preservar as máscaras positivas já corrigidas manualmente;
- recriar a pasta `images/` a partir das imagens originais;
- gerar automaticamente máscaras pretas para as imagens negativas;
- criar um novo `dataset_segmentacao.csv`;
- executar verificações para impedir IDs positivos entre os negativos.


## 1. Bibliotecas


In [10]:
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio


## 2. Caminhos e configurações


In [11]:
# Arquivo mestre da triagem corrigida
CAMINHO_TRIAGEM = Path(
    "../data/processed/triagem_manual.csv"
)

# Imagens Sentinel-2 originais
PASTA_IMAGENS_ORIGINAIS = Path(
    "../data/raw/images"
)

# Estrutura do dataset de segmentação
PASTA_SEGMENTACAO = Path(
    "../data/segmentacao"
)

PASTA_IMAGENS = (
    PASTA_SEGMENTACAO / "images"
)

# As máscaras positivas já foram corrigidas manualmente
PASTA_MASCARAS_POSITIVAS = (
    PASTA_SEGMENTACAO / "masks" / "positiva"
)

# Esta pasta será recriada automaticamente
PASTA_MASCARAS_NEGATIVAS = (
    PASTA_SEGMENTACAO / "masks" / "negativa"
)

CAMINHO_DATASET = Path(
    "../data/processed/dataset_segmentacao.csv"
)

# Proporção por imagem:
# 1 positiva : 3 negativas
RAZAO_NEGATIVAS = 3

SEMENTE = 42

print("Triagem:", CAMINHO_TRIAGEM.resolve())
print("Imagens originais:", PASTA_IMAGENS_ORIGINAIS.resolve())
print("Máscaras positivas:", PASTA_MASCARAS_POSITIVAS.resolve())


Triagem: /home/gabriel-david/Banco_de_areia/data/processed/triagem_manual.csv
Imagens originais: /home/gabriel-david/Banco_de_areia/data/raw/images
Máscaras positivas: /home/gabriel-david/Banco_de_areia/data/segmentacao/masks/positiva


## 3. Funções auxiliares


In [12]:
def normalizar_id(valor):
    """Converte IDs lidos do CSV para uma string limpa."""

    return (
        str(valor)
        .strip()
        .replace(".0", "")
    )


def converter_bool(serie):
    """Converte bool/string do CSV para booleano de forma segura."""

    if serie.dtype == bool:
        return serie

    return (
        serie.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False
        })
    )


## 4. Carregar e padronizar a triagem


In [13]:
if not CAMINHO_TRIAGEM.exists():
    raise FileNotFoundError(
        "Arquivo de triagem não encontrado:\n"
        f"{CAMINHO_TRIAGEM.resolve()}"
    )

triagem = pd.read_csv(
    CAMINHO_TRIAGEM
)

triagem["id"] = (
    triagem["id"]
    .apply(normalizar_id)
)

triagem["possui_banco"] = converter_bool(
    triagem["possui_banco"]
)

triagem["usar_dataset"] = converter_bool(
    triagem["usar_dataset"]
)

triagem["qualidade"] = (
    triagem["qualidade"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print(
    f"Total da triagem: {len(triagem)}"
)

display(
    triagem.head()
)


Total da triagem: 2396


,id,possui_banco,qualidade,usar_dataset,observacao
0,1725,False,boa,True,corrigido para não possui banco
1,1726,False,boa,True,sem banco
2,1727,False,boa,True,sem banco
3,1728,False,boa,True,sem banco
4,1729,False,boa,True,sem banco


## 5. Selecionar imagens válidas


In [14]:
boas = triagem[
    (triagem["qualidade"] == "boa")
    & (triagem["usar_dataset"] == True)
].copy()

positivas = boas[
    boas["possui_banco"] == True
].copy()

negativas_disponiveis = boas[
    boas["possui_banco"] == False
].copy()

ids_positivos = set(
    positivas["id"]
)

# Segurança adicional:
# nenhum positivo pode permanecer no conjunto negativo
negativas_disponiveis = (
    negativas_disponiveis[
        ~negativas_disponiveis["id"].isin(
            ids_positivos
        )
    ]
    .copy()
)

print(f"Imagens boas: {len(boas)}")
print(f"Positivas: {len(positivas)}")
print(
    "Negativas disponíveis:",
    len(negativas_disponiveis)
)

intersecao = (
    ids_positivos
    & set(negativas_disponiveis["id"])
)

print(
    "Conflitos positivo/negativo:",
    len(intersecao)
)

if intersecao:
    raise ValueError(
        "Ainda existem IDs classificados "
        "simultaneamente como positivos e negativos:\n"
        f"{sorted(intersecao)}"
    )


Imagens boas: 422
Positivas: 50
Negativas disponíveis: 372
Conflitos positivo/negativo: 0


## 6. Conferir as máscaras positivas antes de continuar


In [15]:
if not PASTA_MASCARAS_POSITIVAS.exists():
    raise FileNotFoundError(
        "A pasta de máscaras positivas não foi encontrada:\n"
        f"{PASTA_MASCARAS_POSITIVAS.resolve()}"
    )

ids_mascaras_positivas = {
    arquivo.stem
    for arquivo
    in PASTA_MASCARAS_POSITIVAS.glob("*.tif")
}

faltando_mascara_positiva = sorted(
    ids_positivos
    - ids_mascaras_positivas
)

mascaras_positivas_extras = sorted(
    ids_mascaras_positivas
    - ids_positivos
)

print(
    "Máscaras positivas encontradas:",
    len(ids_mascaras_positivas)
)

print(
    "Positivas sem máscara:",
    len(faltando_mascara_positiva)
)

if faltando_mascara_positiva:
    print(
        "IDs sem máscara positiva:",
        faltando_mascara_positiva
    )

print(
    "Máscaras positivas que não aparecem "
    "como positivas na triagem:",
    len(mascaras_positivas_extras)
)

if mascaras_positivas_extras:
    print(
        mascaras_positivas_extras
    )

if faltando_mascara_positiva:
    raise ValueError(
        "Existem imagens positivas sem máscara final. "
        "Corrija isso antes de montar o dataset."
    )


Máscaras positivas encontradas: 50
Positivas sem máscara: 0
Máscaras positivas que não aparecem como positivas na triagem: 0


## 7. Selecionar as negativas


In [16]:
N_NEGATIVAS_DESEJADAS = (
    len(positivas)
    * RAZAO_NEGATIVAS
)

N_NEGATIVAS = min(
    N_NEGATIVAS_DESEJADAS,
    len(negativas_disponiveis)
)

negativas_selecionadas = (
    negativas_disponiveis.sample(
        n=N_NEGATIVAS,
        random_state=SEMENTE
    )
    .copy()
)

print(
    f"Positivas: {len(positivas)}"
)

print(
    f"Negativas selecionadas: "
    f"{len(negativas_selecionadas)}"
)

print(
    "Proporção real:",
    f"1:{len(negativas_selecionadas) / len(positivas):.2f}"
)


Positivas: 50
Negativas selecionadas: 150
Proporção real: 1:3.00


## 8. Montar o dataframe final


In [17]:
positivas_dataset = positivas.copy()
positivas_dataset["classe"] = "positiva"

negativas_dataset = (
    negativas_selecionadas.copy()
)

negativas_dataset["classe"] = "negativa"

dataset = pd.concat(
    [
        positivas_dataset,
        negativas_dataset
    ],
    ignore_index=True
)

# Embaralha somente a ordem do CSV.
# Isso não altera a seleção feita acima.
dataset = (
    dataset.sample(
        frac=1,
        random_state=SEMENTE
    )
    .reset_index(drop=True)
)

# Caminhos relativos úteis para os próximos notebooks
dataset["caminho_imagem"] = (
    dataset["id"]
    .apply(
        lambda x:
        f"../data/segmentacao/images/{x}.tif"
    )
)

dataset["caminho_mascara"] = dataset.apply(
    lambda linha: (
        f"../data/segmentacao/masks/"
        f"{linha['classe']}/{linha['id']}.tif"
    ),
    axis=1
)

display(
    dataset[
        [
            "id",
            "classe",
            "possui_banco",
            "caminho_imagem",
            "caminho_mascara"
        ]
    ].head(10)
)

print()
print(
    dataset["classe"].value_counts()
)

print(
    f"Total final: {len(dataset)}"
)


,id,classe,possui_banco,caminho_imagem,caminho_mascara
0,30232,negativa,False,../data/segmentacao/images/30232.tif,../data/segmentacao/masks/negativa/30232.tif
1,13299,positiva,True,../data/segmentacao/images/13299.tif,../data/segmentacao/masks/positiva/13299.tif
2,70685,positiva,True,../data/segmentacao/images/70685.tif,../data/segmentacao/masks/positiva/70685.tif
3,113437,negativa,False,../data/segmentacao/images/113437.tif,../data/segmentacao/masks/negativa/113437.tif
4,137818,negativa,False,../data/segmentacao/images/137818.tif,../data/segmentacao/masks/negativa/137818.tif
5,10234,negativa,False,../data/segmentacao/images/10234.tif,../data/segmentacao/masks/negativa/10234.tif
6,15978,negativa,False,../data/segmentacao/images/15978.tif,../data/segmentacao/masks/negativa/15978.tif
7,111209,negativa,False,../data/segmentacao/images/111209.tif,../data/segmentacao/masks/negativa/111209.tif
8,3504,negativa,False,../data/segmentacao/images/3504.tif,../data/segmentacao/masks/negativa/3504.tif
9,123654,positiva,True,../data/segmentacao/images/123654.tif,../data/segmentacao/masks/positiva/123654.tif



classe
negativa    150
positiva     50
Name: count, dtype: int64
Total final: 200


## 9. Recriar a pasta de imagens

A pasta `images/` será reconstruída do zero com apenas as imagens que fazem parte do novo dataset.

As imagens originais em `data/raw/images/` **não são alteradas**.


In [18]:
# Recria apenas a pasta de imagens do dataset.
# NÃO mexe nas imagens originais.

if PASTA_IMAGENS.exists():
    shutil.rmtree(
        PASTA_IMAGENS
    )

PASTA_IMAGENS.mkdir(
    parents=True,
    exist_ok=True
)

arquivos_ausentes = []

for id_imagem in dataset["id"]:

    origem = (
        PASTA_IMAGENS_ORIGINAIS
        / f"{id_imagem}.tif"
    )

    destino = (
        PASTA_IMAGENS
        / f"{id_imagem}.tif"
    )

    if origem.exists():
        shutil.copy2(
            origem,
            destino
        )

    else:
        arquivos_ausentes.append(
            id_imagem
        )

print(
    f"Imagens copiadas: "
    f"{len(list(PASTA_IMAGENS.glob('*.tif')))}"
)

print(
    f"Imagens ausentes: "
    f"{len(arquivos_ausentes)}"
)

if arquivos_ausentes:
    print(
        arquivos_ausentes
    )

    raise FileNotFoundError(
        "Algumas imagens do dataset não foram "
        "encontradas na pasta de imagens originais."
    )


Imagens copiadas: 200
Imagens ausentes: 0


## 10. Recriar as máscaras negativas

Todas as imagens negativas recebem uma máscara totalmente preta:

- `0` = fundo;
- nenhum pixel positivo.

A pasta de máscaras positivas **não é modificada**.


In [19]:
# Apaga e recria SOMENTE as máscaras negativas.
# As máscaras positivas permanecem intactas.

if PASTA_MASCARAS_NEGATIVAS.exists():
    shutil.rmtree(
        PASTA_MASCARAS_NEGATIVAS
    )

PASTA_MASCARAS_NEGATIVAS.mkdir(
    parents=True,
    exist_ok=True
)

for id_imagem in negativas_selecionadas["id"]:

    caminho_imagem = (
        PASTA_IMAGENS
        / f"{id_imagem}.tif"
    )

    caminho_mascara = (
        PASTA_MASCARAS_NEGATIVAS
        / f"{id_imagem}.tif"
    )

    with rasterio.open(
        caminho_imagem
    ) as src:

        mascara = np.zeros(
            (
                src.height,
                src.width
            ),
            dtype=np.uint8
        )

        perfil = src.profile.copy()

    perfil.update(
        count=1,
        dtype="uint8",
        nodata=0
    )

    with rasterio.open(
        caminho_mascara,
        "w",
        **perfil
    ) as dst:

        dst.write(
            mascara,
            1
        )

print(
    "Máscaras negativas criadas:",
    len(
        list(
            PASTA_MASCARAS_NEGATIVAS.glob(
                "*.tif"
            )
        )
    )
)


Máscaras negativas criadas: 150


## 11. Auditoria final


In [20]:
ids_imagens = {
    arquivo.stem
    for arquivo
    in PASTA_IMAGENS.glob("*.tif")
}

ids_negativos = set(
    negativas_selecionadas["id"]
)

ids_mascaras_negativas = {
    arquivo.stem
    for arquivo
    in PASTA_MASCARAS_NEGATIVAS.glob(
        "*.tif"
    )
}

problemas = []

if len(ids_imagens) != len(dataset):
    problemas.append(
        "Quantidade de imagens diferente "
        "da quantidade de registros do dataset."
    )

if ids_negativos != ids_mascaras_negativas:
    problemas.append(
        "IDs das máscaras negativas não "
        "correspondem aos negativos selecionados."
    )

if ids_positivos & ids_mascaras_negativas:
    problemas.append(
        "Há máscara negativa para uma imagem positiva."
    )

# Confere se cada registro possui imagem e máscara
faltando_imagem = []
faltando_mascara = []

for _, linha in dataset.iterrows():

    id_imagem = linha["id"]
    classe = linha["classe"]

    caminho_imagem = (
        PASTA_IMAGENS
        / f"{id_imagem}.tif"
    )

    caminho_mascara = (
        PASTA_SEGMENTACAO
        / "masks"
        / classe
        / f"{id_imagem}.tif"
    )

    if not caminho_imagem.exists():
        faltando_imagem.append(
            id_imagem
        )

    if not caminho_mascara.exists():
        faltando_mascara.append(
            id_imagem
        )

print("========== AUDITORIA ==========")

print(
    f"Positivas: {len(positivas)}"
)

print(
    f"Negativas: {len(negativas_selecionadas)}"
)

print(
    f"Total: {len(dataset)}"
)

print(
    f"Imagens na pasta: {len(ids_imagens)}"
)

print(
    "Máscaras positivas:",
    len(ids_mascaras_positivas)
)

print(
    "Máscaras negativas:",
    len(ids_mascaras_negativas)
)

print(
    "Imagens faltando:",
    len(faltando_imagem)
)

print(
    "Máscaras faltando:",
    len(faltando_mascara)
)

print(
    "Conflitos positivo/negativo:",
    len(
        ids_positivos
        & ids_mascaras_negativas
    )
)

if faltando_imagem:
    print(
        "IDs sem imagem:",
        faltando_imagem
    )

if faltando_mascara:
    print(
        "IDs sem máscara:",
        faltando_mascara
    )

if problemas:
    print("\nProblemas encontrados:")

    for problema in problemas:
        print("-", problema)

    raise ValueError(
        "A auditoria encontrou inconsistências."
    )

if faltando_imagem or faltando_mascara:
    raise ValueError(
        "Existem arquivos faltando."
    )

print(
    "\nDataset consistente."
)


========== AUDITORIA ==========
Positivas: 50
Negativas: 150
Total: 200
Imagens na pasta: 200
Máscaras positivas: 50
Máscaras negativas: 150
Imagens faltando: 0
Máscaras faltando: 0
Conflitos positivo/negativo: 0

Dataset consistente.


## 12. Salvar o CSV final


In [21]:
dataset.to_csv(
    CAMINHO_DATASET,
    index=False
)

print(
    "Dataset salvo em:"
)

print(
    CAMINHO_DATASET.resolve()
)

print()

print(
    dataset["classe"].value_counts()
)


Dataset salvo em:
/home/gabriel-david/Banco_de_areia/data/processed/dataset_segmentacao.csv

classe
negativa    150
positiva     50
Name: count, dtype: int64
